In [1]:
# https://devocean.sk.com/experts/techBoardDetail.do?ID=165703&boardType=experts&page=&searchData=&subIndex=&idList=&searchText=&techType=&searchDataSub=&searchDataMain=&writerID=automan&comment=

In [ ]:
import torch
from datasets import Dataset, load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    pipeline,
    TrainingArguments,
)
from peft import LoraConfig, PeftModel
from trl import SFTTrainer

In [ ]:
# Base model
MODEL_ID = "google/gemma-2b-it"
base_model = AutoModelForCausalLM.from_pretrained(MODEL_ID, device_map="auto")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.padding_side = "right"

# QLoRA (4bit)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16
)
finetuned_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, device_map="auto", quantization_config=bnb_config
)
lora_config = LoraConfig(
    r=6,
    lora_alpha=8,
    lora_dropout=0.05,
    target_modules=[
        "q_proj",
        "o_proj",
        "k_proj",
        "v_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    task_type="CAUSAL_LM",
)

In [ ]:
from datasets import load_dataset

dataset = load_dataset("daekeun-ml/naver-news-summarization-ko")
dataset

In [ ]:
def generate_prompt(example: dict, tokenize: bool = False) -> str | list[int]:
    messages = [
        {
            "role": "user",
            "content": "다음 글을 요약해주세요:\n\n {}".format(example["document"]),
        },
        {"role": "assistant", "content": "{}".format(example["summary"])},
    ]
    result = tokenizer.apply_chat_template(
        messages, tokenize=tokenize, add_generation_prompt=False
    )
    return result

In [ ]:
output_text = generate_prompt(dataset["train"][0])
print(output_text)

In [ ]:
return

In [ ]:
trainer = SFTTrainer(
    model=base_model,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    max_seq_length=512,
    args=TrainingArguments(
        output_dir="outputs",
        # num_train_epochs = 1,
        # max_steps=3000,
        # warmup_steps=100,
        max_steps=100,
        warmup_steps=10,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=4,
        optim="paged_adamw_8bit",
        learning_rate=2e-4,
        fp16=True,
        logging_steps=10,
        push_to_hub=False,
        report_to="none",
    ),
    peft_config=lora_config,
    formatting_func=generate_prompt,
)

In [ ]:
trainer.train()

In [ ]:
ADAPTER_MODEL = "lora_adapter"
trainer.model.save_pretrained(ADAPTER_MODEL)

In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL, device_map="auto", torch_dtype=torch.float16
)
model = PeftModel.from_pretrained(
    model, ADAPTER_MODEL, device_map="auto", torch_dtype=torch.float16
)
finetuned_model = model.merge_and_unload()
# model.save_pretrained("gemma-2b-it-sum-ko")
pipe_finetuned = pipeline(
    "text-generation", model=finetuned_model, tokenizer=tokenizer, max_new_tokens=512
)

In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL, device_map="auto", torch_dtype=torch.float16
)
pipe = pipeline("text-generation", model=model, tokenizer=tokenizer, max_new_tokens=512)

In [ ]:
doc = dataset["test"]["document"][10]

messages = [{"role": "user", "content": "다음 글을 요약해주세요:\n\n{}".format(doc)}]
prompt = pipe_finetuned.tokenizer.apply_chat_template(
    messages, tokenize=False, add_generation_prompt=True
)

In [ ]:
print(prompt)

In [ ]:
outputs = pipe(
    prompt,
    do_sample=True,
    temperature=0.2,
    top_k=50,
    top_p=0.95,
    add_special_tokens=True,
)
print(outputs[0]["generated_text"][len(prompt) :])

In [ ]:
outputs = pipe_finetuned(
    prompt,
    do_sample=True,
    temperature=0.2,
    top_k=50,
    top_p=0.95,
    add_special_tokens=True,
)
print(outputs[0]["generated_text"][len(prompt) :])